# Human-in-the-Loop (HITL) Email Agent
This notebook demonstrates how to design a LangChain agent using an OpenRouter model with a Human-in-the-Loop (HITL) middleware pattern. It includes tools for drafting and sending emails with review mechanisms for approving, editing, or rejecting critical operations.

In [1]:
# Install the necessary LangChain and OpenRouter packages
!pip install -qU langchain langchain-openrouter

In [2]:
# Retrieve the OpenRouter API key securely from Google Colab's user credentials
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [3]:
# Initialize the OpenRouter model using LangChain's unified chat model factory
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key
)

In [4]:
# Define LangChain tools for both staging (drafting) and executing (sending) email tasks
from langchain.tools import tool

@tool
def draft_email(recipient: str, subject: str, body: str) -> dict:
    """Drafts an email to a specified recipient with a subject and body.

    Args:
        recipient: The email address of the person receiving the email.
        subject: The subject line of the email.
        body: The main content/body text of the email.

    Returns:
        dict: A dictionary containing the structured email details (recipient, subject, body).
    """
    return {
        "recipient": recipient,
        "subject": subject,
        "body": body
    }

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Sends an email to a specified recipient with a subject and body.

    Args:
        recipient: The email address of the person receiving the email.
        subject: The subject line of the email.
        body: The main content/body text of the email.

    Returns:
        str: A confirmation string indicating that the email was sent successfully.
    """
    print(f"Sending email to {recipient} with subject: {subject} and body: {body}")
    return "Email sent successfully!"

In [5]:
# Configure the Human-in-the-Loop (HITL) middleware to intercept critical actions
# In this configuration, 'send_email' requires human approval, edit, or rejection, while 'draft_email' is automatic
from langchain.agents.middleware import HumanInTheLoopMiddleware

human_review = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {
            "allowed_decisions": ["approve", "edit", "reject"]
        },
        "draft_email": False
    },
)

In [6]:
# Construct the final conversational agent integrated with an in-memory memory bank and HITL middleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[draft_email, send_email],
    middleware=[human_review],
    checkpointer=InMemorySaver(),
)

In [7]:
# Set up execution configuration containing a unique session thread ID using UUIDv7
from langchain_core.utils.uuid import uuid7

config = {"configurable": {"thread_id": str(uuid7())}}

In [8]:
# Invoke the agent to start the email dispatch request pipeline
from langchain.messages import HumanMessage

result = agent.invoke(
    {"messages": [HumanMessage("I want to send an email to gboy teling him to get ready for jumat. his email is gody@gmail.com")]},
    config=config,
    version="v2"
)

response = result.value["messages"][-1]
final_response = response.content

In [9]:
# Inspect the response text containing the state of the agent's action
final_response

''

In [10]:
# Display captured interrupts currently blocking execution waiting for human decision
result.interrupts

(Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'recipient': 'gody@gmail.com', 'subject': 'Reminder: Get Ready for Jumat', 'body': "Assalamu alaikum Gboy,\n\nJust a friendly reminder to get ready for Jumat (Friday prayer) today. Don't forget to perform ghusl, wear clean clothes, and head to the mosque early for the khutbah.\n\nMay Allah accept our prayers and grant us blessings on this blessed day.\n\nJazakAllah khair,\nYour friend"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'recipient\': \'gody@gmail.com\', \'subject\': \'Reminder: Get Ready for Jumat\', \'body\': "Assalamu alaikum Gboy,\\n\\nJust a friendly reminder to get ready for Jumat (Friday prayer) today. Don\'t forget to perform ghusl, wear clean clothes, and head to the mosque early for the khutbah.\\n\\nMay Allah accept our prayers and grant us blessings on this blessed day.\\n\\nJazakAllah khair,\\nYour friend"}'}], 'review_configs': [{'action_name': 'send_email', '

In [11]:
# Inspect details of the interrupted actions and metadata for review
for interrupt in result.interrupts:
    print(interrupt.value)

{'action_requests': [{'name': 'send_email', 'args': {'recipient': 'gody@gmail.com', 'subject': 'Reminder: Get Ready for Jumat', 'body': "Assalamu alaikum Gboy,\n\nJust a friendly reminder to get ready for Jumat (Friday prayer) today. Don't forget to perform ghusl, wear clean clothes, and head to the mosque early for the khutbah.\n\nMay Allah accept our prayers and grant us blessings on this blessed day.\n\nJazakAllah khair,\nYour friend"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'recipient\': \'gody@gmail.com\', \'subject\': \'Reminder: Get Ready for Jumat\', \'body\': "Assalamu alaikum Gboy,\\n\\nJust a friendly reminder to get ready for Jumat (Friday prayer) today. Don\'t forget to perform ghusl, wear clean clothes, and head to the mosque early for the khutbah.\\n\\nMay Allah accept our prayers and grant us blessings on this blessed day.\\n\\nJazakAllah khair,\\nYour friend"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions

In [12]:
# Resume the execution by capturing user decision (Edit flow illustrated here)
from langgraph.types import Command

# 1. APPROVAL (Commented Out)
# result = agent.invoke(
#     Command(resume={"decisions": [{"type": "approve"}]}),
#     config=config
# )

# 2. EDIT AND RESUME PIPELINE
if result.interrupts:
    interrupt = result.interrupts[0]

    # Collect revised arguments directly from the terminal or prompt input
    recipient = input("Enter recipient's email: ")
    subject = input("Enter email subject: ")
    body = input("Enter email body: ")

    # Resume the workflow execution by passing the modified arguments back inside the Command structure
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email",
                            "args": {
                                "recipient": recipient,
                                "subject": subject,
                                "body": body
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

# 3. REJECTION (Commented Out)
# result = agent.invoke(
#     Command(resume={"decisions": [{"type": "reject", "message": "I don't want this email sent."}]}),
#     config=config
# )

# 4. CUSTOM AGENT FEEDBACK RESPONSE (Commented Out)
# result = agent.invoke(
#     Command(resume={"decisions": [{"type": "respond", "message": "Use john@example.com instead."}]}),
#     config=config
# )

response = result["messages"][-1]
final_response = response.content

Enter recipient's email: gboy22@gmal.com
Enter email subject: Get ready for juman, ode
Enter email body: Get ready for Jumat bro. I'll be at your hostel in mins
Sending email to gboy22@gmal.com with subject: Get ready for juman, ode and body: Get ready for Jumat bro. I'll be at your hostel in mins


In [13]:
# Retrieve and display the final outputs post-middleware decision resolution
final_response

'The email has been sent successfully! However, I notice the human reviewer modified the details before sending:\n\n**Sent to:** gboy22@gmal.com (note: different email address than requested)\n**Subject:** "Get ready for juman, ode"\n**Body:** "Get ready for Jumat bro. I\'ll be at your hostel in mins"\n\nThe original request was to send to **gody@gmail.com** with a reminder about Jumat preparation. If you\'d like to send a corrected version to the intended recipient, I can help with that.'